# Sentence Embeddings with Word2Vec, Doc2Vec, GloVe, and FastText

In this notebook, we take the **scraped Wikipedia article sentences** and transform them into **feature embeddings** using four major word representation techniques:

- **Word2Vec**
- **Doc2Vec**
- **GloVe**
- **FastText**



In [1]:
!pip install fasttext
!pip install gensim

In [11]:
import re
from pathlib import Path

from bs4 import BeautifulSoup
import urllib.request

import numpy as np
import pandas as pd

import gensim
from gensim.models import KeyedVectors
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import FastText

import fasttext

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
class Config:
  BASE_DIR = Path('/content/drive/MyDrive/TA/machine_learning_2/module_02/solutions')
  MODEL_DIR = BASE_DIR / 'models'

## Grab the article from Wikipedia

In [13]:
# Define URL and headers (pretend to be a real browser)
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Create a request object with headers
req = urllib.request.Request(url, headers=headers)

# Open and read the page
with urllib.request.urlopen(req) as response:
    raw_html = response.read()

# Parse the HTML using BeautifulSoup
article_html = BeautifulSoup(raw_html, "lxml")

# Extract all paragraph tags
article_paragraphs = article_html.find_all("p")

# Combine all text into one string
article_text = ""
for para in article_paragraphs:
    article_text += para.get_text()

# Display first few hundred characters for verification
print(article_text[:500])


Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.[1]
Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.
Natural language pr


## Tokenize to Sentences

In [14]:
# Use a sentence tokenizer
corpus = nltk.sent_tokenize(article_text)
corpus[0]

'\nNatural language processing (NLP) is the processing of natural language information by a computer.'

In [15]:
len(corpus)

35

In [16]:
# remove punctuation and extra whitespaces and lowercase
for i in range(len(corpus )):
    corpus [i] = corpus [i].lower()
    corpus [i] = re.sub(r'\W',' ',corpus [i])
    corpus [i] = re.sub(r'\s+',' ',corpus [i])

In [17]:
corpus[0]

' natural language processing nlp is the processing of natural language information by a computer '

## Word2Vec

**Word2Vec** is a neural embedding model developed by **Google** in 2013.

It learns **word-level embeddings** by predicting a word based on its context (CBOW) or predicting surrounding words given a word (Skip-Gram).  
The result is a vector space where similar words have similar vectors — for example,  
> `vector("king") - vector("man") + vector("woman") ≈ vector("queen")`

**Key idea:** Represents individual *words* based on co-occurrence patterns.  
**Limitation:** Does **not** capture meaning beyond the word level (no sentence or document context).


In [18]:
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format(
    "glove.6B.50d.txt",
    binary=False,
    no_header=True
)

print(model.most_similar(positive=["king","woman"], negative=["man"]))

[('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.759214460849762), ('daughter', 0.7473882436752319), ('elizabeth', 0.7460220456123352), ('princess', 0.7424570322036743), ('kingdom', 0.7337412238121033), ('monarch', 0.721449077129364), ('eldest', 0.7184862494468689), ('widow', 0.7099431157112122)]


In [19]:
import numpy as np
from nltk.tokenize import word_tokenize

def word2vec_embedding(corpus, w2v_model, dimension=50):
    """
    Generate a feature vector for a corpus by averaging 50-dimensional word vectors.
    """

    zero_vector = np.zeros(dimension)
    word_vectors = []

    for sentence in corpus:
        tokens = word_tokenize(sentence)
        vectors = [w2v_model[token] for token in tokens if token in w2v_model.key_to_index]
        word_vectors.extend(vectors)

    return np.mean(word_vectors, axis=0) if word_vectors else zero_vector

In [20]:
# Compute one vector per sentence
sentence_vectors = [word2vec_embedding([sentence], model) for sentence in corpus]

# Create DataFrame
df_word2vec = pd.DataFrame({
    "sentence": corpus,
    "embedding": sentence_vectors
})

df_word2vec.head()

,sentence,embedding
0,natural language processing nlp is the proces...,"[0.17174742, 0.037907142, -0.098579295, 0.4886..."
1,nlp is a subfield of computer science and is c...,"[0.24710743, 0.28977495, -0.036706503, 0.34227..."
2,nlp is also related to information retrieval k...,"[0.2309788, 0.14420614, -0.19777273, 0.1947109..."
3,1 major processing tasks in an nlp system inc...,"[0.0970525, 0.3289825, -0.25082302, 0.26224393..."
4,natural language processing has its roots in t...,"[0.16495332, 0.14250223, -0.39352444, 0.166502..."


## Doc2Vec

**Doc2Vec** (also called *Paragraph Vector*) was introduced by **Google** as an extension of Word2Vec.

While Word2Vec gives embeddings for *words*, Doc2Vec also learns a unique vector for each **document** (or sentence).  
This allows it to represent entire sentences or paragraphs as a single dense vector.

**Key idea:** Learns a document ID vector alongside word vectors during training.  
**Use case:** Great for representing **sentences**, **paragraphs**, or **articles** as fixed-size feature vectors.


In [21]:
# Tag each sentence/document
# Each document must have a unique tag (ID)
tagged_data = [TaggedDocument(words=word_tokenize(doc), tags=[str(i)]) for i, doc in enumerate(corpus)]

In [22]:
# Train the Doc2Vec model
model = Doc2Vec(
    vector_size=100,     # Embedding dimension
    window=5,            # Context window size
    min_count=1,         # Ignore words with total freq lower than this
    workers=4,           # Number of CPU cores to use
    epochs=40,           # Number of training iterations
    dm=1                 # 1 = Distributed Memory (PV-DM), 0 = Distributed Bag of Words (PV-DBOW)
)

model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

In [23]:
# Extract vectors for each document
doc_vectors = [model.dv[str(i)] for i in range(len(tagged_data))]
doc_vectors[0]

array([-0.01536305, -0.00414523, -0.01882127, -0.00664997,  0.00026911,
       -0.01776959,  0.01894585,  0.02897868, -0.04528838, -0.00149513,
       -0.00671136, -0.02560681,  0.0015498 , -0.00459298, -0.00281783,
       -0.03938949,  0.02000922, -0.00908182, -0.02719136, -0.04179534,
        0.00851553, -0.01719573,  0.00934724,  0.00362481,  0.00129371,
       -0.01002095, -0.04201264, -0.02677729, -0.03102594, -0.02326063,
        0.04450956,  0.03594894,  0.01581384,  0.01243744,  0.00868538,
        0.01057465, -0.00911677, -0.0397201 ,  0.00053163, -0.02025574,
        0.02445154, -0.02356699, -0.00901991, -0.03499411,  0.02105092,
       -0.01063272, -0.00518377, -0.00539666,  0.02847032,  0.01814616,
       -0.01179602, -0.02167388,  0.0112487 , -0.01909434, -0.0164261 ,
        0.02267374, -0.0096751 , -0.00304119, -0.0242655 ,  0.00932828,
        0.00644622,  0.02842268,  0.00711195,  0.00314822, -0.00952146,
        0.01052865,  0.04328691,  0.01285717, -0.01557304,  0.02

In [25]:
# Store in a DataFrame
df_doc2vec = pd.DataFrame({
    "sentence": corpus,
    "embedding": doc_vectors
})

df_doc2vec.head()

,sentence,embedding
0,natural language processing nlp is the proces...,"[-0.015363054, -0.0041452264, -0.018821271, -0..."
1,nlp is a subfield of computer science and is c...,"[-0.035898432, 0.002915734, -0.030574325, -0.0..."
2,nlp is also related to information retrieval k...,"[-0.052902725, -0.00087160047, -0.025784587, -..."
3,1 major processing tasks in an nlp system inc...,"[-0.057383873, 0.017688226, -0.040479876, -0.0..."
4,natural language processing has its roots in t...,"[-0.02764413, -0.0036129109, -0.021555513, -0...."


In [26]:
# To infer a new document vector (not in training data)
new_doc = "the weather is nice today"
new_vector = model.infer_vector(word_tokenize(new_doc))
new_vector

array([ 1.7294147e-03, -3.2697795e-03, -4.4082883e-03,  1.2890653e-03,
       -1.7576354e-03,  2.4740135e-03,  6.9233333e-04,  3.9533717e-03,
       -6.0985424e-03, -5.0541195e-03, -3.3549753e-03, -8.4313005e-03,
        2.6849988e-03,  3.7434986e-03, -3.9265570e-03, -1.3116481e-03,
        4.3836464e-03, -3.1305715e-03, -6.9823121e-03, -6.2125586e-03,
       -1.4285613e-03, -5.0181830e-03,  6.3540358e-03,  4.3862904e-03,
       -5.6288028e-03,  3.7885722e-04, -5.1474590e-03, -2.6733861e-03,
       -7.9677446e-04,  1.6281472e-03,  3.8829123e-04,  5.8722490e-04,
        4.4189333e-03, -4.4973305e-04,  3.1722866e-03,  2.3937779e-03,
        3.3099989e-03, -7.5173848e-03, -1.9031302e-03, -6.4125760e-03,
        3.7503599e-03, -3.6873919e-04, -2.9642790e-04, -7.3602065e-03,
        6.5012518e-03, -2.9578812e-03,  2.6028845e-03, -1.9549399e-03,
        5.7591842e-03, -1.5179013e-04, -4.2916194e-04, -2.2744939e-03,
        1.0481706e-03, -4.5882370e-03, -3.3804949e-03,  3.7393477e-03,
      

## Glove

**GloVe (Global Vectors for Word Representation)** was developed by **Stanford University**.

Instead of predicting words like Word2Vec, GloVe is a **count-based model** that learns embeddings from the **global co-occurrence matrix** of words in a corpus.  
It captures both local context (like Word2Vec) and global statistical information (like matrix factorization).

**Key idea:** Combines the efficiency of Word2Vec with the statistical strength of matrix factorization.  
**Use case:** Pre-trained on massive corpora (Wikipedia, Common Crawl), making it ideal for general-purpose text embeddings.


In [28]:
def load_glove_model(glove_file):
    """
    Load a pre-trained GloVe embedding file into a Python dictionary.

    Args:
        glove_file (str): Path to the GloVe text file.

    Returns:
        dict: Mapping of words to their embedding vectors (numpy arrays).
    """
    glove_model = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_model[word] = vector
    return glove_model

In [29]:
glove_model = load_glove_model("glove.6B.50d.txt")

In [30]:
def get_glove_embedding(corpus, glove_model, vector_size=100):
    """
    Generate an average GloVe embedding for a corpus (list of sentences).

    Args:
        corpus (list of str): List of cleaned sentences.
        glove_model (dict): Dictionary mapping words to their GloVe embeddings.
        vector_size (int): Dimension of GloVe vectors (e.g., 50, 100, 200, 300).

    Returns:
        np.ndarray: Averaged GloVe vector for the corpus.
    """
    vectors = []

    for sentence in corpus:
        tokens = word_tokenize(sentence)
        for token in tokens:
            if token in glove_model:
                vectors.append(glove_model[token])

    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)

In [31]:
sentence_vectors = [get_glove_embedding([sentence], glove_model, vector_size=100) for sentence in corpus]

df_glove = pd.DataFrame({
    "sentence": corpus,
    "embedding": sentence_vectors
})

df_glove.head()

,sentence,embedding
0,natural language processing nlp is the proces...,"[0.17174742, 0.037907142, -0.098579295, 0.4886..."
1,nlp is a subfield of computer science and is c...,"[0.24710743, 0.28977495, -0.036706503, 0.34227..."
2,nlp is also related to information retrieval k...,"[0.2309788, 0.14420614, -0.19777273, 0.1947109..."
3,1 major processing tasks in an nlp system inc...,"[0.0970525, 0.3289825, -0.25082302, 0.26224393..."
4,natural language processing has its roots in t...,"[0.16495332, 0.14250223, -0.39352444, 0.166502..."


## FastText

**FastText** was developed by **Facebook AI Research (FAIR)** as an improvement over Word2Vec.

Instead of treating each word as a single token, FastText represents words as **bags of character n-grams** (subword units).  
This allows it to create vectors even for **out-of-vocabulary (OOV)** words by composing them from known subwords.

**Key idea:** Builds embeddings from character-level n-grams → handles misspellings, rare words, and morphologically rich languages better.  
*Use case:** Excellent for noisy or multilingual datasets (e.g., social media text).

In [45]:
# !wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz

[('prince', 0.8236179351806641), ('queen', 0.7839043140411377), ('ii', 0.7746230363845825), ('emperor', 0.7736247777938843), ('son', 0.766719400882721)]


In [26]:
# !gunzip cc.en.300.bin.gz

In [32]:
# ft_model = gensim.models.fasttext.load_facebook_vectors("cc.en.300.bin")
import pandas as pd
import gensim

ft_model = gensim.models.fasttext.load_facebook_vectors("cc.en.300.bin")

In [33]:
corpus = [
    "natural language processing is fascinating",
    "nlp is a subfield of computer science",
    "information retrieval and machine learning are related to nlp",
    "language models help computers understand text"
]

In [34]:
# exp = ft_model["cat"]
# exp

exp = ft_model["cat"]
exp

array([ 0.08105576, -0.02083234, -0.03326922,  0.28555283,  0.13959414,
       -0.1977245 ,  0.10128298,  0.01085356, -0.103824  ,  0.04313416,
       -0.14833796, -0.16765352, -0.15447043, -0.14154345,  0.12743813,
        0.2279076 ,  0.07685639, -0.13873424, -0.20190817,  0.01528534,
       -0.06999817,  0.11306947,  0.01669297,  0.11389008,  0.02094817,
       -0.31620952,  0.09814467, -0.1449248 ,  0.09949644,  0.2211973 ,
        0.02225026,  0.06751259, -0.06465218,  0.11267239, -0.0256991 ,
       -0.04765478,  0.03917777,  0.00168321, -0.11691307, -0.27667975,
       -0.06021226,  0.11350961, -0.11300616,  0.08379158, -0.21970375,
        0.06771149,  0.0296645 , -0.05783203, -0.12882547,  0.09360313,
       -0.0628323 , -0.08581617,  0.17381558, -0.10044617, -0.28967732,
       -0.01837742,  0.01613754, -0.0155128 , -0.11910667,  0.20571907,
        0.2338278 ,  0.17166924,  0.07774843,  0.05795193, -0.05462614,
       -0.05604232,  0.07913449,  0.32939437, -0.21045874, -0.11

In [35]:
import numpy as np
from nltk.tokenize import word_tokenize

def get_fasttext_embedding(corpus, ft_model):
    """
    Generate an average FastText embedding for a corpus (list of sentences).
    """
    vectors = []

    for sentence in corpus:
        tokens = word_tokenize(sentence)
        for token in tokens:
            vectors.append(ft_model[token])

    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(ft_model.vector_size)

In [37]:
sentence_vectors = [
    get_fasttext_embedding([sentence], ft_model) for sentence in corpus
]

df_fasttext = pd.DataFrame({
    "sentence": corpus,
    "embedding": sentence_vectors
})

df_fasttext.head()

,sentence,embedding
0,natural language processing is fascinating,"[-0.015364654, -0.042854153, -0.018912178, 0.0..."
1,nlp is a subfield of computer science,"[-0.022881249, -0.12465323, -0.02924503, -0.08..."
2,information retrieval and machine learning are...,"[-0.026420543, 0.024097959, 0.01944297, 0.0297..."
3,language models help computers understand text,"[-0.0053157546, 0.016437309, 0.015198707, 0.03..."
